In [1]:
import torch
import numpy as np
from torch.utils.data import DataLoader
from torch import nn

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# --- 1. POSITIONAL EMBEDDING ---
class SinusoidalPosEmb(nn.Module):
    def __init__(self, dim=256, max_positions=10000): # FastSpeech 2 default encoder dim is 256
        super(SinusoidalPosEmb, self).__init__()
        self.dim = dim
        self.max_positions = max_positions

    def forward(self, x):
        # Generates positions based on the sequence length of the incoming tensor x
        batch_size, seq_len, _ = x.shape
        positions = torch.arange(seq_len, dtype=torch.float32, device=x.device)
        
        half_dim = self.dim // 2
        emb = torch.log(torch.tensor(self.max_positions)) / (half_dim - 1)
        frequencies = torch.arange(half_dim, dtype=torch.float32, device=x.device)
        frequencies = torch.exp(frequencies * -emb)
        angle_matrix = positions[..., None] * frequencies[None, :]
        
        pos_embeddings = torch.cat([torch.sin(angle_matrix), torch.cos(angle_matrix)], dim=-1)
        # Broadcast across batch dimension: [Batch, Seq_Len, dim]
        return pos_embeddings.unsqueeze(0).repeat(batch_size, 1, 1)


# --- 2. ENCODER / DECODER CORE BLOCK ---
class FFTBlock(nn.Module):
    def __init__(self, dim, heads=2, inner_dim=1024, kernel_size=[9, 1], dropout=0.1):
        super(FFTBlock, self).__init__()
        self.mha = nn.MultiheadAttention(embed_dim=dim, num_heads=heads, dropout=dropout, batch_first=True)
        self.norm1 = nn.LayerNorm(dim)
        
        self.conv1 = nn.Conv1d(in_channels=dim, out_channels=inner_dim, kernel_size=kernel_size[0], padding=(kernel_size[0] - 1) // 2)
        self.conv2 = nn.Conv1d(in_channels=inner_dim, out_channels=dim, kernel_size=kernel_size[1], padding=(kernel_size[1] - 1) // 2)
        
        self.norm2 = nn.LayerNorm(dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        residual = x
        attn_out, _ = self.mha(query=x, key=x, value=x, key_padding_mask=mask)
        x = self.norm1(residual + self.dropout(attn_out))

        residual = x
        x = x.transpose(1, 2)
        x = F.relu(self.conv1(x))
        x = self.conv2(x)
        x = x.transpose(1, 2)
        
        x = self.norm2(residual + self.dropout(x))
        return x


# --- 3. VARIANCE MODULES ---
class VariancePredictor(nn.Module):
    def __init__(self, dim, inner_dim=256, dropout=0.5):
        super(VariancePredictor, self).__init__()
        self.conv1 = nn.Conv1d(in_channels=dim, out_channels=inner_dim, kernel_size=3, stride=1, padding=1)
        self.conv2 = nn.Conv1d(in_channels=inner_dim, out_channels=inner_dim, kernel_size=3, stride=1, padding=1)
        self.norm1 = nn.LayerNorm(inner_dim)
        self.norm2 = nn.LayerNorm(inner_dim)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
        self.linear = nn.Linear(inner_dim, 1)

    def forward(self, x):
        x = x.transpose(1, 2)
        x = self.conv1(x)
        x = x.transpose(1, 2)
        x = self.relu(x)
        x = self.norm1(x)
        x = self.dropout(x)

        x = x.transpose(1, 2)
        x = self.conv2(x) 
        x = x.transpose(1, 2)
        x = self.relu(x)
        x = self.norm2(x) 
        x = self.dropout(x)

        x = self.linear(x)
        return x.squeeze(-1)


class VarianceAdapter(nn.Module):
    def __init__(self, dim, pitch_bins=256, energy_bins=256):
        super(VarianceAdapter, self).__init__()
        self.dim = dim
        self.duration_predictor = VariancePredictor(self.dim)
        self.pitch_predictor = VariancePredictor(self.dim)
        self.energy_predictor = VariancePredictor(self.dim)
        self.pitch_embedding = nn.Embedding(pitch_bins, self.dim)
        self.energy_embedding = nn.Embedding(energy_bins, self.dim)

    def forward(self, x, target_duration=None, target_pitch=None, target_energy=None, alpha=1.0, max_len=None):
        pred_duration = self.duration_predictor(x)
        
        if target_duration is not None:
            # FIXED COMMENT: Training Mode (Ground-Truth Forcing)
            x = self.length_regulator(x, target_duration, alpha=1.0, max_len=max_len)
        else:
            # FIXED COMMENT: Inference Mode (Using Model Predictions)
            est_duration = torch.clamp(torch.exp(pred_duration) - 1, min=0)
            x = self.length_regulator(x, est_duration, alpha=alpha, max_len=None)

        pred_pitch = self.pitch_predictor(x)
        pitch_source = target_pitch if target_pitch is not None else pred_pitch
        pitch_source_binned = torch.clamp(pitch_source.long(), 0, 255)
        x = x + self.pitch_embedding(pitch_source_binned)

        pred_energy = self.energy_predictor(x)
        energy_source = target_energy if target_energy is not None else pred_energy
        energy_source_binned = torch.clamp(energy_source.long(), 0, 255)
        x = x + self.energy_embedding(energy_source_binned)

        return x, pred_duration, pred_pitch, pred_energy

    def length_regulator(self, h_pho, durations, alpha=1.0, max_len=None):
        durations = torch.clamp(durations * alpha, min=0).long()
        output = []
        for i in range(h_pho.size(0)):
            expanded = torch.repeat_interleave(h_pho[i], durations[i], dim=0)
            output.append(expanded)
            
        output = nn.utils.rnn.pad_sequence(output, batch_first=True)
        if max_len is not None:
            output = self.pad_or_crop(output, max_len)
        return output

    def pad_or_crop(self, x, max_len):
        current_len = x.size(1)
        if current_len < max_len:
            padding = (0, 0, 0, max_len - current_len)
            x = torch.nn.functional.pad(x, padding, "constant", 0)
        elif current_len > max_len:
            x = x[:, :max_len, :]
        return x


# --- 4. THE DECODER CLASS ---
class FastSpeech2Decoder(nn.Module):
    def __init__(self, d_model=384, n_layers=4, n_head=2, d_inner=1024, n_mel_channels=80, dropout=0.1):
        super(FastSpeech2Decoder, self).__init__()
        
        self.layer_stack = nn.ModuleList([
            FFTBlock(dim=d_model, heads=n_head, inner_dim=d_inner, kernel_size=[9, 1], dropout=dropout)
            for _ in range(n_layers)
        ])
        self.mel_linear = nn.Linear(d_model, n_mel_channels)

    def forward(self, x, mask=None):
        for decoder_layer in self.layer_stack:
            x = decoder_layer(x, mask=mask)
        
        # Project from hidden dimensions to final acoustic mel channels (e.g., 384 -> 80)
        mel_output = self.mel_linear(x)
        return mel_output


# --- 5. UNIFIED COMPLETE MODEL ---
class FastSpeech2(nn.Module):
    def __init__(self, vocab_size=256, enc_dim=256, dec_dim=384, n_mel_channels=80):
        super(FastSpeech2, self).__init__()
        
        # Encoder Embeddings
        self.phoneme_embedding = nn.Embedding(vocab_size, enc_dim)
        self.encoder_pos_emb = SinusoidalPosEmb(dim=enc_dim)
        
        # Encoder Block Stack (4 layers is standard in FastSpeech 2)
        self.encoder_stack = nn.ModuleList([FFTBlock(dim=enc_dim) for _ in range(4)])
        
        # Variance Adapter Pipeline
        self.variance_adapter = VarianceAdapter(dim=enc_dim)
        
        # Decoder Dimension Bridge: Seamlessly scales features up from enc_dim (256) to dec_dim (384)
        self.decoder_bridge = nn.Linear(enc_dim, dec_dim)
        
        # Decoder Embeddings & Stack
        self.decoder_pos_emb = SinusoidalPosEmb(dim=dec_dim)
        self.decoder = FastSpeech2Decoder(d_model=dec_dim, n_mel_channels=n_mel_channels)

    def forward(self, phoneme_ids, target_duration=None, target_pitch=None, target_energy=None, alpha=1.0, max_len=None):
        """
        phoneme_ids: Tensor of token integers [Batch, Phoneme_Len]
        """
        # 1. Text processing + Encoder Positional Encodings
        x = self.phoneme_embedding(phoneme_ids)
        x = x + self.encoder_pos_emb(x)
        
        # 2. Forward pass through Encoder Stack
        for enc_layer in self.encoder_stack:
            x = enc_layer(x)
            
        # 3. Variance Adaptation (converts text timeline -> frame audio timeline)
        x, pred_dur, pred_pitch, pred_energy = self.variance_adapter(
            x, target_duration, target_pitch, target_energy, alpha, max_len
        )
        
        # 4. Upscale vector sizing from 256 to 384 for the wider Decoder pipeline
        x = self.decoder_bridge(x)
        
        # 5. Append frame-level positional tracking
        x = x + self.decoder_pos_emb(x)
        
        # 6. Decode into raw acoustic features
        mel_output = self.decoder(x)
        
        # Returns the target output along with prediction tensors to compute training losses
        return mel_output, pred_dur, pred_pitch, pred_energy

In [3]:
if __name__ == "__main__":
    # Instantiate the network
    model = FastSpeech2()
    print("FastSpeech 2 Architecture Successfully Built!")

    # Simulate a training batch of 2 sentences, containing 12 phoneme tokens each
    dummy_phoneme_batch = torch.randint(1, 255, (2, 12)) 
    
    # Run Inference (predictions handle lengths dynamically)
    mel_output, pred_dur, pred_pitch, pred_energy = model(dummy_phoneme_batch, alpha=1.0)
    
    print("\n--- Output Dimensional Verification ---")
    print("Generated Spectrogram Shape [Batch, Frame_Len, Mel_Channels]:", mel_output.shape)
    print("Predicted Durations Shape   [Batch, Phoneme_Len]:           ", pred_dur.shape)

FastSpeech 2 Architecture Successfully Built!

--- Output Dimensional Verification ---
Generated Spectrogram Shape [Batch, Frame_Len, Mel_Channels]: torch.Size([2, 12, 80])
Predicted Durations Shape   [Batch, Phoneme_Len]:            torch.Size([2, 12])
